# Stage 1: Full-Dataset Random Forest

The Random Forest pipeline developed on the Stage 1 development dataset is applied to the full available INSTANCE sample dataset.

The full dataset contains the available earthquake and noise recordings after applying the waveform selection and windowing criteria established during dataset preparation.

The same feature extraction procedure and Random Forest configuration selected during development are used for the full-data experiment.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Add project root to the Python path
project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.features import build_feature_matrix

## Loading the full Stage 1 dataset

The preprocessed full-data waveform splits generated during Stage 1 dataset preparation are loaded.

The train, validation, and test partitions are kept unchanged for the Random Forest experiment.

In [2]:
processed_dir = Path("../data/processed")

X_train_full = np.load(
    processed_dir / "stage1_full_train_X.npy"
)

y_train_full = np.load(
    processed_dir / "stage1_full_train_y.npy"
)

X_val_full = np.load(
    processed_dir / "stage1_full_validation_X.npy"
)

y_val_full = np.load(
    processed_dir / "stage1_full_validation_y.npy"
)

X_test_full = np.load(
    processed_dir / "stage1_full_test_X.npy"
)

y_test_full = np.load(
    processed_dir / "stage1_full_test_y.npy"
)

print("X_train_full:", X_train_full.shape)
print("X_val_full:  ", X_val_full.shape)
print("X_test_full: ", X_test_full.shape)

print("y_train_full:", y_train_full.shape)
print("y_val_full:  ", y_val_full.shape)
print("y_test_full: ", y_test_full.shape)

X_train_full: (7461, 3, 2000)
X_val_full:   (1743, 3, 2000)
X_test_full:  (1796, 3, 2000)
y_train_full: (7461,)
y_val_full:   (1743,)
y_test_full:  (1796,)


## Extracting features for the full-data Random Forest

The shared feature extraction implementation is applied to the full training, validation, and test waveform sets.

The resulting feature matrices contain the same 15 features used during the development experiment.

In [3]:
X_train_features_full = build_feature_matrix(
    X_train_full,
    fs=100.0
)

X_val_features_full = build_feature_matrix(
    X_val_full,
    fs=100.0
)

X_test_features_full = build_feature_matrix(
    X_test_full,
    fs=100.0
)

print(
    "Training features:",
    X_train_features_full.shape
)

print(
    "Validation features:",
    X_val_features_full.shape
)

print(
    "Test features:",
    X_test_features_full.shape
)

Training features: (7461, 15)
Validation features: (1743, 15)
Test features: (1796, 15)


## Selecting the Random Forest features

The same 15-feature set used during development is retained for the full-data experiment to keep the feature representation consistent between experiments.

In [4]:
selected_features = [
    "E_rms",
    "E_peak",
    "E_energy",
    "E_dominant_frequency",
    "E_spectral_centroid",

    "N_rms",
    "N_peak",
    "N_energy",
    "N_dominant_frequency",
    "N_spectral_centroid",

    "Z_rms",
    "Z_peak",
    "Z_energy",
    "Z_dominant_frequency",
    "Z_spectral_centroid"
]

X_train_rf_full = X_train_features_full[selected_features]
X_val_rf_full = X_val_features_full[selected_features]
X_test_rf_full = X_test_features_full[selected_features]

print("Full training RF matrix:", X_train_rf_full.shape)

Full training RF matrix: (7461, 15)


## Training the full-data Random Forest

The Random Forest is trained using the full training split and the selected feature set.

The hyperparameter configuration established during the development experiment is retained for this full-data run.

In [5]:
best_rf_full = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

best_rf_full.fit(
    X_train_rf_full,
    y_train_full
)

print("Full-data Random Forest training complete.")

Full-data Random Forest training complete.


## Evaluating the full-data Random Forest

Validation performance is evaluated before the final test evaluation.

In [6]:
y_val_rf_full = best_rf_full.predict(X_val_rf_full)

print("Validation Accuracy:", accuracy_score(y_val_full, y_val_rf_full))
print("Validation Precision:", precision_score(
    y_val_full, y_val_rf_full, zero_division=0
))
print("Validation Recall:", recall_score(
    y_val_full, y_val_rf_full, zero_division=0
))
print("Validation F1:", f1_score(
    y_val_full, y_val_rf_full, zero_division=0
))

Validation Accuracy: 0.9403327596098681
Validation Precision: 0.9444776119402986
Validation Recall: 0.9930947897049592
Validation F1: 0.9681762545899633


## Final test evaluation

The selected Random Forest is evaluated on the held-out test set.

The test set has not been used for model fitting or model selection and is used to obtain the final performance estimate for the full-data experiment.

In [7]:
y_test_rf_full = best_rf_full.predict(X_test_rf_full)

print("Test Accuracy:",
      accuracy_score(y_test_full, y_test_rf_full))

print("Test Precision:",
      precision_score(
          y_test_full,
          y_test_rf_full,
          zero_division=0
      ))

print("Test Recall:",
      recall_score(
          y_test_full,
          y_test_rf_full,
          zero_division=0
      ))

print("Test F1:",
      f1_score(
          y_test_full,
          y_test_rf_full,
          zero_division=0
      ))

print("\nConfusion matrix:")
print(confusion_matrix(y_test_full, y_test_rf_full))

Test Accuracy: 0.9476614699331849
Test Precision: 0.9485549132947977
Test Recall: 0.996962332928311
Test F1: 0.9721563981042654

Confusion matrix:
[[  61   89]
 [   5 1641]]


## Decision threshold analysis

Random Forest probability outputs are evaluated at multiple decision thresholds to examine the trade-off between earthquake detection and false alarms.

In [8]:
thresholds = [0.3, 0.5, 0.7, 0.9]

test_probabilities = best_rf_full.predict_proba(
    X_test_rf_full
)[:, 1]

for threshold in thresholds:

    y_pred = (
        test_probabilities >= threshold
    ).astype(int)

    print(
        f"Threshold = {threshold:.1f} | "
        f"Accuracy = {accuracy_score(y_test_full, y_pred):.3f} | "
        f"Precision = {precision_score(y_test_full, y_pred, zero_division=0):.3f} | "
        f"Recall = {recall_score(y_test_full, y_pred, zero_division=0):.3f} | "
        f"F1 = {f1_score(y_test_full, y_pred, zero_division=0):.3f}"
    )

Threshold = 0.3 | Accuracy = 0.925 | Precision = 0.926 | Recall = 0.998 | F1 = 0.961
Threshold = 0.5 | Accuracy = 0.948 | Precision = 0.949 | Recall = 0.997 | F1 = 0.972
Threshold = 0.7 | Accuracy = 0.960 | Precision = 0.975 | Recall = 0.981 | F1 = 0.978
Threshold = 0.9 | Accuracy = 0.847 | Precision = 0.994 | Recall = 0.838 | F1 = 0.909


## Selecting the classification threshold

The decision threshold is selected using validation-set performance. The selected threshold is then fixed before evaluating the model on the held-out test set.

In [9]:
validation_probabilities = best_rf_full.predict_proba(
    X_val_rf_full
)[:, 1]

thresholds = np.arange(0.1, 1.0, 0.05)

threshold_results = []

for threshold in thresholds:

    y_pred = (
        validation_probabilities >= threshold
    ).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "accuracy": accuracy_score(y_val_full, y_pred),
        "precision": precision_score(
            y_val_full, y_pred, zero_division=0
        ),
        "recall": recall_score(
            y_val_full, y_pred, zero_division=0
        ),
        "f1": f1_score(
            y_val_full, y_pred, zero_division=0
        )
    })

threshold_results = pd.DataFrame(threshold_results)

display(
    threshold_results.sort_values(
        "f1",
        ascending=False
    ).head(10)
)

,threshold,accuracy,precision,recall,f1
12,0.70,0.947791,0.971734,0.971124,0.971429
10,0.60,0.945496,0.955596,0.986190,0.970652
11,0.65,0.944923,0.964618,0.975518,0.970037
9,0.55,0.942054,0.948857,0.989956,0.968971
8,0.50,0.940333,0.944478,0.993095,0.968176
13,0.75,0.940333,0.976328,0.957941,0.967047
7,0.45,0.935743,0.936873,0.996861,0.965937
6,0.40,0.931727,0.931499,0.998745,0.963950
5,0.35,0.927711,0.927199,0.999372,0.961934
4,0.30,0.923695,0.922943,1.000000,0.959928


## Final test-set evaluation

The classification threshold is fixed at 0.70 based on validation-set performance. The held-out test set is evaluated using this threshold without further model or threshold selection.

In [10]:
final_threshold = 0.70

test_probabilities = best_rf_full.predict_proba(
    X_test_rf_full
)[:, 1]

y_test_pred_final = (
    test_probabilities >= final_threshold
).astype(int)

test_accuracy = accuracy_score(y_test_full, y_test_pred_final)
test_precision = precision_score(
    y_test_full,
    y_test_pred_final,
    zero_division=0
)
test_recall = recall_score(
    y_test_full,
    y_test_pred_final,
    zero_division=0
)
test_f1 = f1_score(
    y_test_full,
    y_test_pred_final,
    zero_division=0
)

print(f"Threshold: {final_threshold:.2f}")
print(f"Accuracy:  {test_accuracy:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall:    {test_recall:.4f}")
print(f"F1-score:  {test_f1:.4f}")

cm = confusion_matrix(
    y_test_full,
    y_test_pred_final
)

print("\nConfusion matrix:")
print(cm)

Threshold: 0.70
Accuracy:  0.9599
Precision: 0.9752
Recall:    0.9812
F1-score:  0.9782

Confusion matrix:
[[ 109   41]
 [  31 1615]]
